In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [3]:
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='lexical'
)

In [4]:
X = dataloader.df.drop(['url', 'type'], axis=1, inplace=False)
y = dataloader.df['type']

X_train, X_test, y_train, y_test = dataloader.train_test_split(X, y, train_split=0.8, random_state=None)

In [5]:
#{'poisson', 'squared_error', 'absolute_error', 'friedman_mse'}
#{'0.771', '0.777', 'NA' ,'0.777'} Best one is squared

rf_hyperparams = {
    'n_estimators': 100,
    'criterion': 'squared_error', 
    'max_depth': 10
}

In [11]:
pipe = Pipeline(
    [
        ('stdscaler', StandardScaler()), 
        #('onehotencoder', OneHotEncoder(**onehot_hyperparams))
        ('randfor', RandomForestRegressor(**rf_hyperparams))
    ]
)

rFModel = pipe.fit(X_train, y_train).score(X_test, y_test)

In [8]:
import treelite.sklearn

In [14]:
model = pipe.named_steps['randfor']

treelite_model = treelite.sklearn.import_model(model)
treelite_model.serialize('model.tl')